# Adaptive Retrieval-Augmented Generation (RAG) System

## Overview

This system implements an advanced Retrieval-Augmented Generation (RAG) approach that adapts its retrieval strategy based on the type of query. By leveraging Language Models (LLMs) at various stages, it aims to provide more accurate, relevant, and context-aware responses to user queries.

## Key Components

1. **Query Classifier**: Determines the type of query (Factual, Analytical, Opinion, or Contextual).
2. **Adaptive Retrieval Strategies**: Four distinct strategies tailored to different query types.
3. **LLM Integration**: LLMs are used throughout the process to enhance retrieval and ranking.
4. **Response Generation**: Generates the final response using the retrieved documents as context.

<div style="text-align: center;">

<img src="../images/adaptive_retrieval.svg" alt="adaptive retrieval" style="width:100%; height:auto;">
</div>

## Package Installation

In [ ]:
!pip install faiss-cpu langchain-core langchain-community langchain-openai langchain-text-splitters python-dotenv rank-bm25 PyMuPDF

## Imports & Configuration

Shared config is in `config.py` (project root). Edit that file to change LLM/Embedding for the entire project.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path(os.getcwd()).parent))

from config import get_llm, get_embeddings, check_connections

from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import Dict, Any, List
from pydantic import BaseModel, Field, ConfigDict

In [ ]:
# Verify config and test connections to LLM & Embedding services
check_connections()


### Define the query classifier class

In [ ]:
class categories_options(BaseModel):
    category: str = Field(
        description="The category of the query, the options are: Factual, Analytical, Opinion, or Contextual",
        examples=["Factual"]
    )


class QueryClassifier:
    def __init__(self):
        self.llm = get_llm()
        self.prompt = PromptTemplate(
            input_variables=["query"],
            template="""Classify the following query into exactly one of these categories: Factual, Analytical, Opinion, or Contextual.
Respond with ONLY the category name, nothing else.

Query: {query}
Category:"""
        )
        self.chain = self.prompt | self.llm

    def classify(self, query):
        print("Classifying query...")
        result = self.chain.invoke({"query": query}).content.strip()
        # Parse the category from the response
        for cat in ["Factual", "Analytical", "Opinion", "Contextual"]:
            if cat.lower() in result.lower():
                print(f"Category: {cat}")
                return cat
        print(f"Defaulting to Factual (got: {result})")
        return "Factual"

### Define the Base Retriever class

In [ ]:
class BaseRetrievalStrategy:
    def __init__(self, texts):
        self.embeddings = get_embeddings()
        text_splitter = CharacterTextSplitter(chunk_size=800, chunk_overlap=0)
        self.documents = text_splitter.create_documents(texts)
        self.db = FAISS.from_documents(self.documents, self.embeddings)
        self.llm = get_llm()

    def retrieve(self, query, k=4):
        return self.db.similarity_search(query, k=k)

### Define Factual retriever strategy

In [ ]:
class FactualRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=4):
        print("Retrieving factual...")
        # Use LLM to enhance the query
        enhanced_query_prompt = PromptTemplate(
            input_variables=["query"],
            template="Enhance this factual query for better information retrieval. Return only the enhanced query:\n{query}"
        )
        query_chain = enhanced_query_prompt | self.llm
        enhanced_query = query_chain.invoke({"query": query}).content.strip()
        print(f"Enhanced query: {enhanced_query}")

        # Retrieve documents using the enhanced query
        docs = self.db.similarity_search(enhanced_query, k=k*2)

        # Use LLM to rank the relevance of retrieved documents
        ranking_prompt = PromptTemplate(
            input_variables=["query", "doc"],
            template="""On a scale of 1-10, how relevant is this document to the query?
Respond with ONLY a number between 1 and 10.

Query: '{query}'
Document: {doc}
Score:"""
        )
        ranking_chain = ranking_prompt | self.llm

        ranked_docs = []
        print("Ranking docs...")
        for doc in docs:
            input_data = {"query": enhanced_query, "doc": doc.page_content}
            try:
                score_text = ranking_chain.invoke(input_data).content.strip()
                score = float(''.join(c for c in score_text if c.isdigit() or c == '.')[:4])
            except (ValueError, IndexError):
                score = 5.0
            ranked_docs.append((doc, score))

        ranked_docs.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked_docs[:k]]

### Define Analytical retriever strategy

In [ ]:
class AnalyticalRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=4):
        print("Retrieving analytical...")
        # Use LLM to generate sub-queries
        sub_queries_prompt = PromptTemplate(
            input_variables=["query", "k"],
            template="""Generate exactly {k} sub-questions for comprehensive analysis of this query.
Return each sub-question on a new line, numbered 1-{k}.

Query: {query}
Sub-questions:"""
        )
        sub_queries_chain = sub_queries_prompt | self.llm
        result = sub_queries_chain.invoke({"query": query, "k": k}).content.strip()
        
        # Parse sub-queries from response
        sub_queries = [line.strip().lstrip('0123456789.-) ') for line in result.split('\n') if line.strip()]
        sub_queries = sub_queries[:k]  # Limit to k
        print(f"Sub queries: {sub_queries}")

        all_docs = []
        for sub_query in sub_queries:
            all_docs.extend(self.db.similarity_search(sub_query, k=2))

        # Deduplicate and return top k
        seen = set()
        unique_docs = []
        for doc in all_docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_docs.append(doc)
        
        return unique_docs[:k]

### Define Opinion retriever strategy

In [ ]:
class OpinionRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=3):
        print("Retrieving opinion...")
        # Use LLM to identify potential viewpoints
        viewpoints_prompt = PromptTemplate(
            input_variables=["query", "k"],
            template="""Identify {k} distinct viewpoints or perspectives on this topic.
Return each viewpoint on a new line.

Topic: {query}
Viewpoints:"""
        )
        viewpoints_chain = viewpoints_prompt | self.llm
        result = viewpoints_chain.invoke({"query": query, "k": k}).content.strip()
        viewpoints = [line.strip().lstrip('0123456789.-) ') for line in result.split('\n') if line.strip()]
        print(f"Viewpoints: {viewpoints}")

        all_docs = []
        for viewpoint in viewpoints:
            all_docs.extend(self.db.similarity_search(f"{query} {viewpoint}", k=2))

        # Deduplicate and return top k
        seen = set()
        unique_docs = []
        for doc in all_docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_docs.append(doc)
        
        return unique_docs[:k]

### Define Contextual retriever strategy

In [ ]:
class ContextualRetrievalStrategy(BaseRetrievalStrategy):
    def retrieve(self, query, k=4, user_context=None):
        print("Retrieving contextual...")
        # Use LLM to incorporate user context into the query
        context_prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="""Given the user context: {context}
Reformulate the query to best address the user's needs. Return only the reformulated query.

Original query: {query}
Reformulated query:"""
        )
        context_chain = context_prompt | self.llm
        input_data = {"query": query, "context": user_context or "No specific context provided"}
        contextualized_query = context_chain.invoke(input_data).content.strip()
        print(f"Contextualized query: {contextualized_query}")

        # Retrieve documents using the contextualized query
        docs = self.db.similarity_search(contextualized_query, k=k*2)

        # Use LLM to rank relevance
        ranking_prompt = PromptTemplate(
            input_variables=["query", "context", "doc"],
            template="""Rate the relevance of this document on a scale of 1-10.
Respond with ONLY a number between 1 and 10.

Query: '{query}'
User context: '{context}'
Document: {doc}
Score:"""
        )
        ranking_chain = ranking_prompt | self.llm
        print("Ranking docs...")

        ranked_docs = []
        for doc in docs:
            input_data = {
                "query": contextualized_query,
                "context": user_context or "No specific context provided",
                "doc": doc.page_content
            }
            try:
                score_text = ranking_chain.invoke(input_data).content.strip()
                score = float(''.join(c for c in score_text if c.isdigit() or c == '.')[:4])
            except (ValueError, IndexError):
                score = 5.0
            ranked_docs.append((doc, score))

        ranked_docs.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked_docs[:k]]

### Define the Adaptive Retriever class

In [ ]:
class AdaptiveRetriever:
    def __init__(self, texts: List[str]):
        self.classifier = QueryClassifier()
        self.strategies = {
            "Factual": FactualRetrievalStrategy(texts),
            "Analytical": AnalyticalRetrievalStrategy(texts),
            "Opinion": OpinionRetrievalStrategy(texts),
            "Contextual": ContextualRetrievalStrategy(texts)
        }

    def get_relevant_documents(self, query: str) -> List[Document]:
        category = self.classifier.classify(query)
        strategy = self.strategies[category]
        return strategy.retrieve(query)

### Define LangChain-compatible retriever wrapper

In [ ]:
class PydanticAdaptiveRetriever(BaseRetriever):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    adaptive_retriever: AdaptiveRetriever = Field(exclude=True)

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        return self.adaptive_retriever.get_relevant_documents(query)

    async def _aget_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        return self._get_relevant_documents(query)

### Define the Adaptive RAG class

In [ ]:
class AdaptiveRAG:
    def __init__(self, texts: List[str]):
        adaptive_retriever = AdaptiveRetriever(texts)
        self.retriever = PydanticAdaptiveRetriever(adaptive_retriever=adaptive_retriever)
        self.llm = get_llm()
        
        prompt_template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Answer:"""
        prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
        self.llm_chain = prompt | self.llm

    def answer(self, query: str) -> str:
        docs = self.retriever.invoke(query)
        input_data = {"context": "\n".join([doc.page_content for doc in docs]), "question": query}
        return self.llm_chain.invoke(input_data)

### Demo

In [ ]:
# Usage
texts = [
    "The Earth is the third planet from the Sun and the only astronomical object known to harbor life.",
    "The average distance between the Earth and the Sun is about 150 million kilometers (93 million miles).",
    "Earth's atmosphere is composed of 78% nitrogen, 21% oxygen, and trace amounts of other gases.",
    "The Earth has a magnetic field that protects it from solar radiation and cosmic rays.",
    "Life on Earth began approximately 3.8 billion years ago in the oceans.",
    "There are multiple theories about the origin of life: panspermia, RNA world hypothesis, and deep-sea vent theory.",
    "Climate change is primarily driven by greenhouse gas emissions from human activities.",
    "Some scientists argue that natural climate cycles also play a role in current warming trends.",
]

rag_system = AdaptiveRAG(texts)
print("RAG system initialized successfully!")

### Showcase the four different types of queries

In [ ]:
# Factual query
print("=" * 60)
print("FACTUAL QUERY")
print("=" * 60)
factual_result = rag_system.answer("What is the distance between the Earth and the Sun?")
print(f"Answer: {factual_result.content}\n")

In [ ]:
# Analytical query
print("=" * 60)
print("ANALYTICAL QUERY")
print("=" * 60)
analytical_result = rag_system.answer("How does the Earth's distance from the Sun affect its climate?")
print(f"Answer: {analytical_result.content}\n")

In [ ]:
# Opinion query
print("=" * 60)
print("OPINION QUERY")
print("=" * 60)
opinion_result = rag_system.answer("What are the different theories about the origin of life on Earth?")
print(f"Answer: {opinion_result.content}\n")

In [ ]:
# Contextual query
print("=" * 60)
print("CONTEXTUAL QUERY")
print("=" * 60)
contextual_result = rag_system.answer("How does the Earth's position in the Solar System influence its habitability?")
print(f"Answer: {contextual_result.content}")